# Freeze the P4b task-segmented training schedule

Attach **exact version 1** of the private dataset `thestonedape/task-aware-eeg2text-task-segmented-protocol`, enable Internet, and enable the private Kaggle secret `GITHUB_TOKEN`. Use a CPU session. This notebook clean-remount verifies the sealed parent protocol, freezes the common 40-epoch schedule shared by all three P4b arms, and deeply decodes every generated schedule index. It never loads EEG/text vectors, trains a model, or accesses validation/test. A PASS authorizes only the separately bounded two-batch, three-arm smoke; full training remains unauthorized.

In [ ]:
REPO_URL = 'https://github.com/thestonedape/task-aware-eeg2text.git'
SCHEDULE_COMMIT = '93a5bdcdd4a1fc6b140097921906968459b10fe9'
WORKTREE = '/kaggle/working/SemKey'
OUTPUT = '/kaggle/working/task-aware-eeg2text-task-segmented-schedule'
DATASET_SLUG = 'thestonedape/task-aware-eeg2text-task-segmented-protocol'
PRESERVED_DATASET_VERSION = 1
PRESERVED_SOURCE_ID = 'kaggle-dataset-thestonedape-task-aware-eeg2text-task-segmented-protocol-version-1'
EXPECTED_PARENT_CONTRACT_SHA256 = '396670afc0244cb601364ff89df53944c4f63402191a9d120e6e2648e5baed3b'
EXPECTED_PARENT_REPORT_SHA256 = 'f99ff4ad371e30b86dc9582bb4c35854f6bba863d7868a5f424f93776eab8116'
EXPECTED_PARENT_METADATA_SHA256 = '766c836c96c2025805d81bbbad7d3378faf14b8eca4fc1885653add5a1f35dc9'
EXPECTED_PARENT_VERIFICATION_SHA256 = '376fedd2ca89189653a6a4e784195411bea061c63b220a8dfce33cba0b8b4b32'
EXPECTED_PARENT_VERIFICATION_BYTES = 2851
EXPECTED_SCHEDULE_CONTRACT_SHA256 = 'a6ea34388cd98380654f413b1440d0d5cee0b8065555b0c04a53d0db6ea12287'
EXPECTED_CATALOG_ROWS = 9011
EXPECTED_GLOBAL_BATCHES_PER_EPOCH = 105
EXPECTED_SHAPE = [15, 40, 105, 64]
EXPECTED_CORE_SHA256 = {
    'trial_catalog.csv': '3d93e0cea4290ac22e8111760241d04109392e6f024abae85a4d9504fc4f8fc9',
    'schedule_indices.u32le': '79543e72f496ee3f7a8140556b274c15ecc5992e900a07bed5e5c74a2ddd7cbc',
    'schedule_units.csv': 'e2644a51ac578b18388ce82d07d910714182cf674e60b0a5f2c547067b8720aa',
    'schedule_audit.csv': 'f18299fbba6725f7eb33c4686bdd496a83de139c76f3494f48f6a1148d169fbf',
    'task_segmented_training_schedule_manifest.json': '0cf2a752b5f0a67e7282bc0b4551b4792ceb3d346dd441fef6c359515885270a',
    'task_segmented_training_schedule_report.json': '5e7d953a8ca31e48d6acbd6168b146d695af449ef0ad5bc573a37d306cb7250f',
}
assert len(SCHEDULE_COMMIT) == 40
assert PRESERVED_DATASET_VERSION == 1
assert EXPECTED_SHAPE == [15, 40, EXPECTED_GLOBAL_BATCHES_PER_EPOCH, 64]
assert all(len(value) == 64 for value in (EXPECTED_PARENT_CONTRACT_SHA256, EXPECTED_PARENT_REPORT_SHA256, EXPECTED_PARENT_METADATA_SHA256, EXPECTED_PARENT_VERIFICATION_SHA256, EXPECTED_SCHEDULE_CONTRACT_SHA256, *EXPECTED_CORE_SHA256.values()))

In [ ]:
import glob, hashlib, json, os, platform, shutil, subprocess, sys
from pathlib import Path
from kaggle_secrets import UserSecretsClient

def digest(path):
    state = hashlib.sha256()
    with open(path, 'rb') as handle:
        for block in iter(lambda: handle.read(1024 * 1024), b''):
            state.update(block)
    return state.hexdigest()

github_token = UserSecretsClient().get_secret('GITHUB_TOKEN')
assert github_token, 'Enable the private Kaggle Secret named GITHUB_TOKEN'
askpass = '/kaggle/working/git_askpass.py'
with open(askpass, 'w', encoding='utf-8', newline='\n') as handle:
    handle.write("#!/usr/bin/env python3\nimport os, sys\nprompt = sys.argv[1] if len(sys.argv) > 1 else ''\nprint(os.environ['GITHUB_TOKEN'] if 'Password' in prompt else 'x-access-token')\n")
os.chmod(askpass, 0o700)
clone_env = os.environ.copy()
clone_env.update({'GIT_ASKPASS': askpass, 'GIT_TERMINAL_PROMPT': '0', 'GITHUB_TOKEN': github_token})
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
try:
    subprocess.run(['git', 'clone', REPO_URL, WORKTREE], check=True, env=clone_env)
finally:
    if os.path.exists(askpass):
        os.remove(askpass)
    del github_token, clone_env
subprocess.run(['git', '-C', WORKTREE, 'checkout', '--detach', SCHEDULE_COMMIT], check=True)
actual_commit = subprocess.check_output(['git', '-C', WORKTREE, 'rev-parse', 'HEAD'], text=True).strip()
assert actual_commit == SCHEDULE_COMMIT
subprocess.run([
    sys.executable, '-m', 'unittest',
    'evaluation.test_verify_task_segmented_protocol_artifact',
    'evaluation.test_task_segmented_training_schedule',
    'evaluation.test_verify_task_segmented_training_schedule_output',
], check=True, cwd=WORKTREE)
SCHEDULE_CONTRACT = os.path.join(WORKTREE, 'evaluation', 'task_segmented_training_schedule_contract.json')
assert digest(SCHEDULE_CONTRACT) == EXPECTED_SCHEDULE_CONTRACT_SHA256
print({'python': platform.python_version(), 'schedule_commit': actual_commit, 'schedule_contract_sha256': EXPECTED_SCHEDULE_CONTRACT_SHA256, 'regressions': 'PASS'})

In [ ]:
EXPECTED_PARENT_FILES = {
    'batch_grid_feasibility.csv', 'candidate_pools.csv',
    'confirmation_donors.csv', 'outer_split_assignments.csv',
    'protocol_registry.json', 'pseudo_groups.csv',
    'text_group_folds.csv', 'task_segmented_protocol_report.json',
    'protocol_freeze_run_metadata.json', 'task_segmented_objective_contract.json',
}
report_candidates = glob.glob('/kaggle/input/**/task_segmented_protocol_report.json', recursive=True)
artifact_roots = []
for path in report_candidates:
    root = os.path.dirname(path)
    try:
        names = set(os.listdir(root))
    except OSError:
        continue
    if names == EXPECTED_PARENT_FILES and all(os.path.isfile(os.path.join(root, name)) for name in names):
        artifact_roots.append(root)
artifact_roots = sorted(set(artifact_roots))
assert len(artifact_roots) == 1, ('Attach exact version 1 of the one complete task-segmented protocol dataset', artifact_roots, report_candidates)
ARTIFACT_ROOT = artifact_roots[0]
assert digest(os.path.join(ARTIFACT_ROOT, 'task_segmented_objective_contract.json')) == EXPECTED_PARENT_CONTRACT_SHA256
assert digest(os.path.join(ARTIFACT_ROOT, 'task_segmented_protocol_report.json')) == EXPECTED_PARENT_REPORT_SHA256
assert digest(os.path.join(ARTIFACT_ROOT, 'protocol_freeze_run_metadata.json')) == EXPECTED_PARENT_METADATA_SHA256
PARENT_VERIFICATION_PATH = '/kaggle/working/task_segmented_parent_verification.json'
if os.path.exists(PARENT_VERIFICATION_PATH):
    os.remove(PARENT_VERIFICATION_PATH)
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'verify_task_segmented_protocol_artifact.py'),
    '--artifact-root', ARTIFACT_ROOT,
    '--preserved-source-id', PRESERVED_SOURCE_ID,
    '--output-report', PARENT_VERIFICATION_PATH,
], check=True, cwd=WORKTREE)
with open(PARENT_VERIFICATION_PATH, encoding='utf-8') as handle:
    parent_verification = json.load(handle)
assert parent_verification['status'] == 'pass'
assert parent_verification['preserved_source_id'] == PRESERVED_SOURCE_ID
assert parent_verification['contract_sha256'] == EXPECTED_PARENT_CONTRACT_SHA256
assert parent_verification['protocol_report_sha256'] == EXPECTED_PARENT_REPORT_SHA256
assert parent_verification['protocol_freeze_run_metadata_sha256'] == EXPECTED_PARENT_METADATA_SHA256
assert parent_verification['counts']['eligible_rows'] == EXPECTED_CATALOG_ROWS
assert parent_verification['counts']['task_rows'] == {'NR': 4126, 'TSR': 4885}
assert parent_verification['counts']['normalized_text_groups'] == 515
assert parent_verification['counts']['pseudo_task_text_identities'] == 573
assert parent_verification['counts']['candidate_pools'] == 18022
assert digest(PARENT_VERIFICATION_PATH) == EXPECTED_PARENT_VERIFICATION_SHA256
assert os.path.getsize(PARENT_VERIFICATION_PATH) == EXPECTED_PARENT_VERIFICATION_BYTES
assert parent_verification['training_authorized'] is False
assert parent_verification['held_out_test_accessed'] is False
print({'dataset_slug': DATASET_SLUG, 'dataset_version': PRESERVED_DATASET_VERSION, 'artifact_root': ARTIFACT_ROOT, 'parent_verification': 'PASS'})

In [ ]:
if os.path.exists(OUTPUT):
    shutil.rmtree(OUTPUT)
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'freeze_task_segmented_training_schedule.py'),
    '--protocol-root', ARTIFACT_ROOT,
    '--output-root', OUTPUT,
    '--contract', SCHEDULE_CONTRACT,
], check=True, cwd=WORKTREE)
subprocess.run([
    sys.executable,
    os.path.join(WORKTREE, 'evaluation', 'freeze_task_segmented_training_schedule.py'),
    '--output-root', OUTPUT,
    '--contract', SCHEDULE_CONTRACT,
    '--verify-output-only',
], check=True, cwd=WORKTREE)
if WORKTREE not in sys.path:
    sys.path.insert(0, WORKTREE)
from evaluation.verify_task_segmented_training_schedule_output import verify as deep_verify_schedule
deep_verification = deep_verify_schedule(
    Path(OUTPUT),
    EXPECTED_SCHEDULE_CONTRACT_SHA256,
    EXPECTED_PARENT_REPORT_SHA256,
    expected_catalog_rows=EXPECTED_CATALOG_ROWS,
    expected_batches_per_epoch=EXPECTED_GLOBAL_BATCHES_PER_EPOCH,
    expected_shape=EXPECTED_SHAPE,
)
assert deep_verification['status'] == 'pass'
print(json.dumps(deep_verification, indent=2, sort_keys=True))
print('TASK-SEGMENTED TRAINING SCHEDULE OUTPUT VERIFICATION: PASS')

In [ ]:
REPORT_NAME = 'task_segmented_training_schedule_report.json'
MANIFEST_NAME = 'task_segmented_training_schedule_manifest.json'
CORE_FILES = set(EXPECTED_CORE_SHA256)
assert set(os.listdir(OUTPUT)) == CORE_FILES
actual_core_sha256 = {name: digest(os.path.join(OUTPUT, name)) for name in sorted(CORE_FILES)}
assert actual_core_sha256 == {name: EXPECTED_CORE_SHA256[name] for name in sorted(CORE_FILES)}
with open(os.path.join(OUTPUT, REPORT_NAME), encoding='utf-8') as handle:
    report = json.load(handle)
with open(os.path.join(OUTPUT, MANIFEST_NAME), encoding='utf-8') as handle:
    manifest = json.load(handle)
assert report['status'] == manifest['status'] == 'pass'
assert report['schedule_contract_sha256'] == manifest['schedule_contract_sha256'] == EXPECTED_SCHEDULE_CONTRACT_SHA256
assert report['parent_protocol_report_sha256'] == manifest['parent_protocol_report_sha256'] == EXPECTED_PARENT_REPORT_SHA256
assert report['counts'] == {
    'catalog_rows': 9011, 'schedule_units': 15, 'epochs_per_unit': 40,
    'global_batches_per_epoch': 105, 'batch_size': 64,
    'scheduled_uint32_indices': 4032000,
}
assert manifest['shape'] == EXPECTED_SHAPE
assert manifest['dtype'] == '<u4' and manifest['order'] == 'C'
assert len(manifest['unit_order']) == 15
assert manifest['applicable_arms'] == ['global_mixed', 'true_task_segmented', 'pseudo_task_segmented']
assert manifest['arm_axis_absent_from_binary'] is True
assert manifest['same_schedule_across_arms'] is True
assert manifest['bounded_smoke_authorized'] is True
assert manifest['full_training_authorized'] is False
assert manifest['held_out_test_accessed'] is False
assert report['checks'] == {
    'parent_protocol_clean_remount_verified': True,
    'fit_rows_only': True,
    'exact_16_per_task_pseudo_cell': True,
    'globally_unique_normalized_text_per_batch': True,
    'identity_exposure_gap_at_most_one': True,
    'conditional_row_exposure_gap_at_most_one': True,
    'every_fit_row_covered': True,
    'same_schedule_across_arms': True,
    'arm_axis_absent_from_schedule': True,
    'schedule_differs_by_seed': True,
    'vector_array_or_model_score_loaded': False,
    'official_validation_used': False,
    'held_out_test_accessed': False,
}
assert report['authorization']['bounded_smoke_authorized'] is True
assert report['authorization']['full_training_authorized'] is False
assert deep_verification['shape'] == EXPECTED_SHAPE
assert deep_verification['audit_rows'] == 60
assert deep_verification['schedule_report_sha256'] == EXPECTED_CORE_SHA256[REPORT_NAME]
assert deep_verification['schedule_manifest_sha256'] == EXPECTED_CORE_SHA256[MANIFEST_NAME]
shutil.copy2(SCHEDULE_CONTRACT, os.path.join(OUTPUT, 'task_segmented_training_schedule_contract.json'))
shutil.copy2(PARENT_VERIFICATION_PATH, os.path.join(OUTPUT, 'parent_protocol_verification_report.json'))
parent_verification_sha256 = digest(PARENT_VERIFICATION_PATH)
assert parent_verification_sha256 == EXPECTED_PARENT_VERIFICATION_SHA256
run_metadata = {
    'status': 'pass',
    'schema_version': 1,
    'python': platform.python_version(),
    'schedule_commit': actual_commit,
    'parent_dataset_slug': DATASET_SLUG,
    'parent_dataset_version': PRESERVED_DATASET_VERSION,
    'parent_preserved_source_id': PRESERVED_SOURCE_ID,
    'parent_contract_sha256': EXPECTED_PARENT_CONTRACT_SHA256,
    'parent_protocol_report_sha256': EXPECTED_PARENT_REPORT_SHA256,
    'parent_protocol_metadata_sha256': EXPECTED_PARENT_METADATA_SHA256,
    'parent_verification_report_sha256': EXPECTED_PARENT_VERIFICATION_SHA256,
    'schedule_contract_sha256': EXPECTED_SCHEDULE_CONTRACT_SHA256,
    'schedule_manifest_sha256': EXPECTED_CORE_SHA256[MANIFEST_NAME],
    'schedule_report_sha256': EXPECTED_CORE_SHA256[REPORT_NAME],
    'core_file_sha256': actual_core_sha256,
    'catalog_rows': EXPECTED_CATALOG_ROWS,
    'global_batches_per_epoch': EXPECTED_GLOBAL_BATCHES_PER_EPOCH,
    'shape': EXPECTED_SHAPE,
    'bounded_smoke_authorized': True,
    'full_training_authorized': False,
    'held_out_test_accessed': False,
}
metadata_path = os.path.join(OUTPUT, 'schedule_freeze_run_metadata.json')
metadata_bytes = (json.dumps(run_metadata, indent=2, sort_keys=True) + '\n').encode('utf-8')
assert b'\r\n' not in metadata_bytes
with open(metadata_path, 'wb') as handle:
    handle.write(metadata_bytes)
metadata_sha256 = digest(metadata_path)
expected_final_files = CORE_FILES | {
    'task_segmented_training_schedule_contract.json',
    'parent_protocol_verification_report.json',
    'schedule_freeze_run_metadata.json',
}
assert set(os.listdir(OUTPUT)) == expected_final_files
assert digest(os.path.join(OUTPUT, 'task_segmented_training_schedule_contract.json')) == EXPECTED_SCHEDULE_CONTRACT_SHA256
assert digest(os.path.join(OUTPUT, 'parent_protocol_verification_report.json')) == EXPECTED_PARENT_VERIFICATION_SHA256
assert os.path.getsize(os.path.join(OUTPUT, 'parent_protocol_verification_report.json')) == EXPECTED_PARENT_VERIFICATION_BYTES
if os.path.exists(PARENT_VERIFICATION_PATH):
    os.remove(PARENT_VERIFICATION_PATH)
if os.path.exists(WORKTREE):
    shutil.rmtree(WORKTREE)
print({
    'status': report['status'],
    'catalog_rows': report['counts']['catalog_rows'],
    'global_batches_per_epoch': report['counts']['global_batches_per_epoch'],
    'shape': manifest['shape'],
    'schedule_contract_sha256': EXPECTED_SCHEDULE_CONTRACT_SHA256,
    'schedule_manifest_sha256': EXPECTED_CORE_SHA256[MANIFEST_NAME],
    'schedule_report_sha256': EXPECTED_CORE_SHA256[REPORT_NAME],
    'schedule_freeze_run_metadata_sha256': metadata_sha256,
    'bounded_smoke_authorized': report['authorization']['bounded_smoke_authorized'],
    'full_training_authorized': report['authorization']['full_training_authorized'],
})
print('TASK-SEGMENTED TRAINING SCHEDULE FREEZE: PASS')

After the terminal PASS, save `/kaggle/working/task-aware-eeg2text-task-segmented-schedule` as a **new private Kaggle dataset**, preferably named `task-aware-eeg2text-task-segmented-schedule`. Send its dataset slug, immutable version number, `schedule_contract_sha256`, `schedule_manifest_sha256`, `schedule_report_sha256`, and `schedule_freeze_run_metadata_sha256`. This artifact authorizes only the frozen two-batch smoke for `global_mixed`, `true_task_segmented`, and `pseudo_task_segmented`; it does **not** authorize the full 40-epoch scientific fits.